# Explore a flight

Loads the **cleaned** data for one flight from its `processed/` folder. The columns are the standard schema
described in `nasauli/schema.py` (and in each `*_metadata.json`), so this notebook works the same for every
flight, whatever logger format the raw file used.

Setup (once): `pip install -r requirements.txt matplotlib jupyter`, then run this notebook from the repository root
or from `notebooks/`.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sorted(p.name for p in ROOT.glob("*/processed/*_metadata.json"))

Pick a flight folder and session. Re-run `python -m nasauli process <folder>` first if you changed its raw data.

In [ ]:
FLIGHT = "Tarot450_20260925_165053Z"
meta_path = next((ROOT / FLIGHT / "processed").glob("*_metadata.json"))
stem = meta_path.name.removesuffix("_metadata.json")
meta = json.loads(meta_path.read_text())

df = pd.read_parquet(ROOT / FLIGHT / "processed" / f"{stem}.parquet")
wind_path = ROOT / FLIGHT / "processed" / f"{stem}_wind.parquet"
wind = pd.read_parquet(wind_path) if wind_path.exists() else None
print(stem, df.shape, "wind rows:", None if wind is None else len(wind))
meta["summary"]

Data-quality findings from the pipeline:

In [ ]:
for c in meta["checks"]:
    print(f"[{c['level']}] {c['title']}: {c['detail']}")

Column units and descriptions:

In [ ]:
pd.DataFrame(meta["columns"]).T

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
ax[0].plot(df.elapsed_s, df.alt_rel_m); ax[0].set_ylabel("alt rel [m]")
ax[1].plot(df.elapsed_s, df.groundspeed_m_s, label="groundspeed")
ax[1].plot(df.elapsed_s, df.climb_m_s, label="climb"); ax[1].set_ylabel("m/s"); ax[1].legend()
ax[2].plot(df.elapsed_s, df.roll_deg, label="roll")
ax[2].plot(df.elapsed_s, df.pitch_deg, label="pitch"); ax[2].set_ylabel("deg"); ax[2].legend()
ax[-1].set_xlabel("elapsed [s]")
for a in ax:
    for t0, t1, mode in meta["summary"]["modes"]:
        if mode == "AUTO":
            a.axvspan(t0, t1, color="C0", alpha=0.06)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df.east_m, df.north_m, c=df.elapsed_s, s=2)
plt.axis("equal"); plt.xlabel("east [m]"); plt.ylabel("north [m]"); plt.colorbar(label="elapsed [s]")

Wind sensor, already trimmed to this flight and on the same `elapsed_s` axis:

In [ ]:
if wind is not None and len(wind):
    fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
    ax[0].plot(wind.elapsed_s, wind.wind_speed_m_s); ax[0].set_ylabel("wind [m/s]")
    ax[1].plot(wind.elapsed_s, wind.wind_dir_deg, ".", ms=2); ax[1].set_ylabel("direction [deg]")
    ax[1].set_xlabel("elapsed [s]"); plt.tight_layout()
else:
    print("No wind data overlaps this flight.")

Wind at each flight sample (nearest wind reading within 0.5 s):

In [ ]:
if wind is not None and len(wind):
    joined = pd.merge_asof(df.sort_values("time_gps_utc"), wind[["time_utc", "wind_speed_m_s", "wind_dir_deg"]],
                           left_on="time_gps_utc", right_on="time_utc", direction="nearest",
                           tolerance=pd.Timedelta("0.5s"), suffixes=("", "_wind"))
    joined[["elapsed_s", "groundspeed_m_s", "pitch_deg", "wind_speed_m_s", "wind_dir_deg"]].describe()